# Perspective projection / Perspective imaging
(derived using similar triangles)

- X, Y – object coordinates
- Z – optical axis, it intersects the image at its center
- COP - Center of Projection (optical center (point on the lens where light goes through unchanged), origin of a camera coordinate system)
- The image plane is placed in front of the lense (in contrast to the real imaging system where the sensor is behind the lense) to simplify the math and get not flipped image.
<img src="1_images/perspective_projection.png" width="1000">

Problem: division by the (non-constant) Z is a non-linear transformation.

Note: the transformation is linear if 2 conditions are met:
- f(x+y) = f(x) + f(y);
- f(cx) = cf(x)

If we treat Z as variable (not constant), these conditions are not met.

Trick: add one more coordinate (with value 1, on <i>w</i> in general case)
Converting from homogeneous coordinates:
<img src="1_images/converting_from_homogeneous_coordinates.png" width="1000">

Note: it is scale invariant. If we multiply original coordinate by constant value _a_ the projected coordinates will remain the same.

This allows us to do projection as a matrix multiplication in homogeneous coordinates:
<img src="1_images/projection_matrix.png" width="1000">

Note: (x, y, z/f) → homogeneous coordinates. We can preserve them until we need coordinates in the image coordinate system. Then we transfer to (fx/z, fy/z).

## Vanishing point
Almost all parallel lines intersect after perspective projection (lines parallel to the projection plane (c=0) stay parallel):
<img src="1_images/vanishing_point.png" width="1000">

Note: If we take several different sets of parallel lines on the same plane, their vanishing points will belong to the same line which is called 'horizon'
<img src="1_images/horizon.png" width="1000">

## Other types of projection
1. Orthogonal (orthographic) Projection / Parallel Projection, (x,y,z) -> (x, y)
Projection matrix:
<img src="1_images/orthogonal_projection.png" width="1000">

2. Weak perspective projection. Basic idea: the difference in <i>z</i> coordinate of different objects is not significant compared to the distance to the camera, so we take it as $z_0$.
<img src="1_images/weak_perspective_projection.png" width="1000">

# Stereo geometry

Human ways of depth perception:
1. Perspective effect (we know real object sizes and compare them to sizes we actually see)
2. Shading - the light falls differently on different depth
3. Texture - the same texture looks differently depending on depth
4. Focus/defocus
5. Motion

Basic idea of stereo images: two slightly different images. Human vision system lines up little local pixel areas to make a stereo and then process the image to recognize objects.

Geometry for a simple stereo system (f - focal length, $x_{l}$ is positive, $x_{r}$ is negative)
<img src="1_images/simple_stereo_system.png" width="1200">
Disparity map:
<img src="1_images/disparity_map.png" width="1200">
Evidence: distant objects have lower disparity then close objects. Disparity is invertially proportional to the depth. (Moon follows you!)


# Epipolar geometry

We have 2 cameras filming the same point P (exact location is unknown) from different known positions (O, O').
Given point p on the left image, we need to find a corresponding point p' on the right image. It cannot be anywhere on the image, it will belong to the line l' (so the task is reduced to 1d search).

- Baseline: line joining the camera centers (OO')
- Epipolar plane: plane containing baseline and world point (OO'P)
- Epipolar line: intersection of epipolar plane with the image plane - come in pairs (l, l')
- Epipole: point of intersection of baseline with image plane (e, e'). All epipolar lines intersect in this point (except the case of the case of parallel image planes. In this case epipolar lines are parallel and intersect at infinity)
<img src="1_images/epipolar_geometry.png" width="1200">

# Stereo correspondence

Beyond the hard constraint of epipolar geometry, there are "soft" constraints to help identify correspondence points:
1. Similarity - pixels should look similar on the left and on the right images
2. Uniqueness - there is NO MORE than 1 pixel in the right image mathing the pixel on the left image
3. Ordering - pixels goes A, B, C on the left image go A, B, C on the right image
4. Disparity gradient is limited - disparity doesn't change too quickly

Based on similarity constraint: we could search for left set of pixels on the right image's epipolar line using normalized cross-correlation.
Problem: texture-less window. Small window does not catch details, large window - losses variations in depth of small details
<img src="1_images/cross_correlation_window_size.png" width="1200">

Uniqueness constraint allows no match between pixels on the left and the right image because of the Occlusion problem:
<img src="1_images/occlusion.png" width="800">

Ordering constraint could be violated in the next situations:

<img src="1_images/ordering_constraint1.png" width="800">

Better approaches: estimate correspondence jointly instead of individual correspondence.

1. Scanline method: use Dynamic Programming to match pixels over pair of epipolar lines.
<img src="1_images/scanline.png" width="800">

2. Energy minimization:

<p><img src="1_images/em_data_term.png" width="800"></p>
<p><img src="1_images/em_smoothness_term.png" width="800"></p>
<p><img src="1_images/em_total_energy.png" width="800"></p>

3. Graph cut algorithms

Challenges:
1. Low-contrast; textureless image regions
2. Occlusion
3. Violation of brightness constancy (e.g. specular reflection)
4. Really large baselines
5. Camera calibration errors